# Mesh Studio trên Google Colab — *Aurasoft Systems*
Tạo model 3D từ ảnh (hoặc từ mô tả) bằng GPU của Colab, dùng chế độ **Ultra** (Hunyuan3D-2 bản đầy đủ, 50 bước, lưới 512³).

**Cách dùng:** `Runtime ▸ Change runtime type ▸ T4 GPU`, rồi chạy lần lượt từng ô từ trên xuống (Shift+Enter).

> ⚠️ Phiên miễn phí có thể bị ngắt. Kết quả được lưu vào Google Drive (`MyDrive/MeshStudio`) để không bị mất.
> Lần chạy đầu cần tải model (~5–6 GB), mất vài phút.

## 1) Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > T4 GPU"
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

## 2) Kết nối Google Drive & chọn nơi lưu
`CACHE_ON_DRIVE = True` lưu model tải về trên Drive (~6 GB) để lần sau khỏi tải lại. Đặt `False` nếu Drive sắp đầy.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

CACHE_ON_DRIVE = True
BASE = '/content/drive/MyDrive/MeshStudio'
os.makedirs(BASE, exist_ok=True)

CACHE = f'{BASE}/cache' if CACHE_ON_DRIVE else '/content/cache'
os.environ['HF_HOME'] = f'{CACHE}/huggingface'
os.environ['U2NET_HOME'] = f'{CACHE}/u2net'
os.environ['TORCH_HOME'] = f'{CACHE}/torch'
os.environ['MESH_DATA_DIR'] = f'{BASE}/data'          # kết quả nằm ở MeshStudio/data/outputs
for p in (CACHE, f'{BASE}/data'):
    os.makedirs(p, exist_ok=True)
print('Đã thiết lập thư mục lưu tại:', BASE)

## 3) Tải mã nguồn Mesh Studio
Nếu repo GitHub của bạn để **Private**, hãy chuyển sang Public hoặc dùng token trong `REPO_URL`.

In [ ]:
REPO_URL = 'https://github.com/vinhlephu-lpv/MeshP.git'
!rm -rf /content/MeshP
!git clone --depth 1 {REPO_URL} /content/MeshP

## 4) Cài thư viện (không cài lại PyTorch của Colab)

In [ ]:
%%capture
!pip install -q einops omegaconf pymeshlab rembg onnxruntime trimesh fast-simplification scikit-image scipy \
    moderngl xatlas imageio opencv-python-headless pyyaml tqdm safetensors hf_transfer \
    "huggingface-hub==0.36.2" "transformers==4.45.2" "diffusers==0.32.2" "accelerate==1.2.1"

## 5) Nạp Mesh Studio & kiểm tra
Nếu ô này báo lỗi liên quan `numpy`: `Runtime ▸ Restart session`, rồi chạy lại từ ô 2 (bỏ qua ô 4).

In [ ]:
import sys
sys.path.insert(0, '/content/MeshP/backend')
from mesh_core.config import ENGINES_3D
from mesh_core.pipeline import image_to_asset, text_to_images
from mesh_core.postprocess import PostprocessOptions
print('Các chế độ khả dụng:')
for e in ENGINES_3D:
    print(f" - {e['key']:<18} {e['label']}  [{e['badge']}]")

## 6) Cài đặt chất lượng
- `ENGINE = 'hunyuan3d_ultra'` → đẹp nhất (khoảng 3–6 phút/model trên T4). Có thể đổi sang `'hunyuan3d_char'` (High), `'hunyuan3d_mini_hq'` (Medium)… cho nhanh hơn.

In [ ]:
ENGINE = 'hunyuan3d_ultra'
TARGET_FACES = 200_000     # số tam giác tối đa
PALETTE_COLORS = 1024      # số màu tối đa
HEIGHT_M = 1.0             # chiều cao model trong Unity (mét)

## 7A) Tạo từ ẢNH của bạn
Chọn 1 hoặc nhiều ảnh (nền gì cũng được, nền sẽ tự xoá). Mỗi ảnh nên chỉ có **một** vật thể, nhìn thẳng từ phía trước.

In [ ]:
from google.colab import files
import os
os.makedirs('/content/inputs', exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    open(f'/content/inputs/{name}', 'wb').write(data)
print('Đã nhận:', list(uploaded))

In [ ]:
import glob, gc, torch
from PIL import Image
from pathlib import Path

results = []
for path in sorted(glob.glob('/content/inputs/*')):
    name = Path(path).stem
    print(f'\n=== {name} ===')
    img = Image.open(path).convert('RGB')
    opts = PostprocessOptions(target_faces=TARGET_FACES, palette_colors=PALETTE_COLORS, target_height_m=HEIGHT_M)
    meta = image_to_asset(img, name, 'default', opts, engine=ENGINE)
    results.append(meta)
    gc.collect(); torch.cuda.empty_cache()
print('\nXong', len(results), 'model.')

## 7B) (Tuỳ chọn) Tạo từ MÔ TẢ tiếng Anh
Bước 1 vẽ ảnh 2D bằng Stable Diffusion, bước 2 chọn ảnh ưng ý và dựng 3D.

In [ ]:
SUBJECT = 'a plump orange pumpkin'
imgs = text_to_images(SUBJECT, 'default', num_images=4, seed=None)
from IPython.display import display
for i, im in enumerate(imgs):
    print('Mẫu', i); display(im.resize((256, 256)))

In [ ]:
PICK = 0   # chọn số mẫu ở trên
opts = PostprocessOptions(target_faces=TARGET_FACES, palette_colors=PALETTE_COLORS, target_height_m=HEIGHT_M)
meta = image_to_asset(imgs[PICK], SUBJECT[:30], 'default', opts, engine=ENGINE)
results = [meta]

## 8) Xem kết quả (4 góc nhìn)

In [ ]:
from IPython.display import display, Image as IPImage
for m in results:
    print(m['name'], '|', m['stats'])
    display(IPImage(filename=m['dir'] + '/turnaround.png'))

## 9) Đóng gói & tải về
Mỗi model gồm `.glb`, `.obj`, bảng màu, ảnh xem trước. Bản gốc cũng đã nằm sẵn trong Google Drive.

In [ ]:
import shutil, os
from google.colab import files
zips = []
for m in results:
    z = shutil.make_archive(f"/content/{m['slug']}", 'zip', m['dir'])
    zips.append(z)
    print('Đã nén:', z)
if zips:
    files.download(zips[-1])   # tải gói mới nhất; các gói khác nằm trong /content và trong Drive
print('Toàn bộ kết quả trong Drive:', os.environ['MESH_DATA_DIR'] + '/outputs')